# Advanced RAG PDF AI Analyzer

In [ ]:
!pip install groq pdfplumber python-dotenv httpx sentence-transformers faiss-cpu numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 132.3 MB/s eta 0:00:00


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "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"
os.environ["MISTRAL_API_KEY"] = "NiVXfdsdULYHLvGP0739f4tayJ5R49VC"

print("✅ API Key loaded")

✅ API Key loaded


In [ ]:
import os
import sys
import base64
import textwrap
import numpy as np
import faiss
import httpx
import pdfplumber

from groq import Groq
from sentence_transformers import SentenceTransformer
from google.colab import files

MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

MAX_TOKENS = 4096
MAX_CHARS = 100000

DIVIDER = "─" * 60

BAHASA_OPTIONS = {
    "1": "Indonesia",
    "2": "Inggris",
    "3": "Melayu",
    "4": "Mandarin (Simplified)",
    "5": "Arab",
    "6": "Jepang",
    "7": "Prancis",
    "8": "Spanyol",
}

print("📦 Loading embedding model...")

embedder = SentenceTransformer(EMBED_MODEL)

client = Groq(api_key=os.environ["GROQ_API_KEY"])

current_pdf = {
    "path": None,
    "text": None,
    "pages": None,
    "chunks": None,
    "index": None
}

qa_history = []

print("✅ Setup selesai")

📦 Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Setup selesai


In [ ]:
def wrap_print(text, width=80):
    for para in text.split("\n"):
        if para.strip():
            print(textwrap.fill(para, width=width))
        else:
            print()


def ocr_pdf_online(path):

    api_key = os.environ.get("MISTRAL_API_KEY", "").strip()

    with open(path, "rb") as f:
        pdf_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

    print("📷 OCR processing...")

    response = httpx.post(
        "https://api.mistral.ai/v1/ocr",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": "mistral-ocr-latest",
            "document": {
                "type": "document_url",
                "document_url": f"data:application/pdf;base64,{pdf_b64}",
            },
        },
        timeout=120.0,
    )

    data = response.json()

    pages = data.get("pages", [])

    pages_text = []

    for page in pages:
        text = page.get("markdown", "").strip()

        if text:
            pages_text.append(
                f"[Halaman {page.get('index', 0)+1}]\\n{text}"
            )

    full_text = "\\n\\n".join(pages_text)

    return full_text, len(pages)


def load_pdf(path):

    pages_text = []

    with pdfplumber.open(path) as pdf:

        total_pages = len(pdf.pages)

        for i, page in enumerate(pdf.pages, 1):

            text = page.extract_text()

            if text and text.strip():
                pages_text.append(
                    f"[Halaman {i}]\\n{text.strip()}"
                )

    if not pages_text:
        return ocr_pdf_online(path)

    full_text = "\\n\\n".join(pages_text)

    if len(full_text) > MAX_CHARS:
        full_text = full_text[:MAX_CHARS]

    return full_text, total_pages


def chunk_text(text, chunk_size=1000, overlap=200):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def create_vector_store(chunks):

    print("🧠 Creating embeddings...")

    embeddings = embedder.encode(chunks)

    embeddings = np.array(embeddings, dtype=np.float32)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    return index, embeddings


def retrieve(query, chunks, index, top_k=5):

    query_embedding = embedder.encode([query])

    query_embedding = np.array(query_embedding, dtype=np.float32)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i in indices[0]:
        results.append(chunks[i])

    return results


def call_groq(system_prompt, user_prompt):

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content

print("✅ Function loaded")

✅ Function loaded


## Upload / Reset PDF

In [ ]:
print('\n📂  Pilih file PDF kamu...\n')

uploaded = files.upload()

if uploaded:

    pdf_file = list(uploaded.keys())[0]

    pdf_path = f'/tmp/{pdf_file}'

    with open(pdf_path, 'wb') as f:
        f.write(uploaded[pdf_file])

    print(f'\n⏳  Membaca PDF...')

    try:

        pdf_text, total_pages = load_pdf(pdf_path)

        print("\n🧩 Chunking document...")

        chunks = chunk_text(pdf_text)

        print(f"✅ Total chunks: {len(chunks)}")

        index, embeddings = create_vector_store(chunks)

        current_pdf['path']   = pdf_path
        current_pdf['text']   = pdf_text
        current_pdf['pages']  = total_pages
        current_pdf['chunks'] = chunks
        current_pdf['index']  = index

        qa_history.clear()

        print(f'\n{DIVIDER}')
        print(f'✅  Berhasil memuat PDF!')
        print(f'   📄  File    : {pdf_file}')
        print(f'   📃  Halaman : {total_pages}')
        print(f'   📦  Chunks  : {len(chunks)}')
        print(f'   📊  Teks    : {len(pdf_text):,} karakter')
        print(f'{DIVIDER}')

        print('\n→ Lanjutkan ke Cell fitur yang kamu inginkan.')

    except Exception as e:

        print(f'\n❌  Gagal membaca PDF: {e}')

else:

    print('\n❌  Tidak ada file yang diupload.')


📂  Pilih file PDF kamu...



Saving RKA-KL 2023.pdf to RKA-KL 2023.pdf

⏳  Membaca PDF...

🧩 Chunking document...
✅ Total chunks: 49
🧠 Creating embeddings...

────────────────────────────────────────────────────────────
✅  Berhasil memuat PDF!
   📄  File    : RKA-KL 2023.pdf
   📃  Halaman : 18
   📦  Chunks  : 49
   📊  Teks    : 38,550 karakter
────────────────────────────────────────────────────────────

→ Lanjutkan ke Cell fitur yang kamu inginkan.


## Rangkuman Dokumen

In [ ]:
if current_pdf["text"]:

    system = (
        "Kamu adalah analis dokumen profesional. "
        "Buat rangkuman terstruktur dalam Bahasa Indonesia."
    )

    user = f"""
Dokumen:
{current_pdf['text'][:15000]}

Buat:
1. Gambaran umum
2. Poin penting
3. Kesimpulan
"""

    result = call_groq(system, user)

    print(f"\n{DIVIDER}")
    print("📝 RANGKUMAN DOKUMEN")
    print(DIVIDER)

    wrap_print(result)

else:

    print("❌ Upload PDF terlebih dahulu.")


────────────────────────────────────────────────────────────
📝 RANGKUMAN DOKUMEN
────────────────────────────────────────────────────────────
**Gambaran Umum**

Dokumen Rincian Kertas Kerja Satker T.A. 2023 ini merupakan rencana anggaran dan
kegiatan yang akan dilaksanakan oleh Komisi Pemilihan Umum (KPU) Kabupaten Yapen
Waropen pada tahun 2023. Dokumen ini memuat informasi tentang alokasi anggaran,
kegiatan, dan rincian belanja yang akan dilakukan oleh KPU.

**Poin Penting**

Berikut adalah beberapa poin penting yang dapat ditarik dari dokumen ini:

* Alokasi anggaran untuk KPU Kabupaten Yapen Waropen pada tahun 2023 adalah
sebesar Rp. 33,951,553,000.
* Kegiatan yang akan dilaksanakan oleh KPU meliputi penyelenggaraan pemilu,
pelatihan teknis kepemiluan, sosialisasi/penyuluhan/bimbingan teknis tahapan
pemilu, dan evaluasi dan pelaporan pendaftaran dan verifikasi partai politik
peserta pemilu.
* Belanja yang akan dilakukan oleh KPU meliputi belanja bahan, belanja honor
output kegiatan

## Advanced RAG Q&A

In [ ]:
if current_pdf["text"]:

    query = input("\n❓ Pertanyaan: ").strip()

    relevant_chunks = retrieve(
        query,
        current_pdf["chunks"],
        current_pdf["index"],
        top_k=5
    )

    context = "\n\n".join(relevant_chunks)

    system = (
        "Jawab hanya berdasarkan konteks yang diberikan. "
        "Jika informasi tidak ada, katakan tidak ditemukan. "
        "Jawab dalam Bahasa Indonesia."
    )

    user = f"""
KONTEKS:
{context}

PERTANYAAN:
{query}
"""

    answer = call_groq(system, user)

    qa_history.append({
        "question": query,
        "answer": answer
    })

    print(f"\n{'─'*60}")
    print("💡 JAWABAN:\n")

    wrap_print(answer)

    print(f"{'─'*60}")

else:

    print("❌ Upload PDF terlebih dahulu.")


❓ Pertanyaan: alokasi anggaran terbesar ada di bidang apa? tolong urutkan dari terbesar sampai ke terkecil?

────────────────────────────────────────────────────────────
💡 JAWABAN:

Berdasarkan informasi yang diberikan, alokasi anggaran terbesar ada di bidang
sebagai berikut:

1. 521213 Belanja Honor Output Kegiatan (Rp. 11.879.930.000)
2. 6867.QGE.003 Dukungan Operasional Badan Adhoc (Rp. 4.861.266.000)
3. 521219 Belanja Barang Non Operasional Lainnya (Rp. 3.438.894.000)
4. TJ TAMBAHAN SOSIALISASI (Rp. 228.400.000)
5. TL Tambahan Logistik (Rp. 825.599.000)
6. 6709 Perencanaan Program dan Anggaran, serta Penyusunan Peraturan Pelaksanaan
Penyelenggaraan Pemilu (Rp. 2.173.127.000)
7. 6709.QGE Tata Kelola Kelembagaan Publik Bidang Politik dan Hukum (Rp.
2.165.509.000)
8. 6709.QGE.001 Perencanaan dan Penganggaran Pemilu (Rp. 1.799.607.000)
9. 111 Pelaksanaan Rencana dan Anggaran Pemilu (Rp. 1.378.087.000)
10. 051 Fasilitasi Kesiapan Penyusunan Jumlah Kursi dan Daerah Pemilihan (Rp.
198.49

## Terjemahan Dokumen

In [ ]:
if current_pdf["text"]:

    print("\n🌐 PILIH BAHASA:\n")

    for k, v in BAHASA_OPTIONS.items():
        print(f"[{k}] {v}")

    pilihan = input("\nPilih bahasa: ").strip()

    if pilihan in BAHASA_OPTIONS:

        bahasa = BAHASA_OPTIONS[pilihan]

        system = (
            f"Terjemahkan teks berikut ke Bahasa {bahasa}. "
            "Pertahankan format dan struktur."
        )

        user = current_pdf["text"][:15000]

        result = call_groq(system, user)

        print(f"\n{DIVIDER}")
        print(f"🌐 TERJEMAHAN ({bahasa})")
        print(DIVIDER)

        wrap_print(result)

    else:

        print("❌ Pilihan tidak valid")

else:

    print("❌ Upload PDF terlebih dahulu.")

## Reset Session

In [ ]:
current_pdf = {
    "path": None,
    "text": None,
    "pages": None,
    "chunks": None,
    "index": None
}

qa_history.clear()

print("✅ Session berhasil di-reset")
print("→ Upload PDF baru untuk memulai kembali.")